In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load preprocessed data
X_train = joblib.load('../models/X_train_scaled.joblib')
X_test = joblib.load('../models/X_test_scaled.joblib')
y_train = joblib.load('../models/y_train.joblib')
y_test = joblib.load('../models/y_test.joblib')

print(f"Data loaded successfully")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

Data loaded successfully
X_train: (1178361, 15), X_test: (294591, 15)


In [9]:
# Baseline model
print("=== Training Logistic Regression ===")
lr_model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_lr):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_lr):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_lr):.4f}")
print(f"\n{classification_report(y_test, y_pred_lr)}")

=== Training Logistic Regression ===
Accuracy: 0.9548
Precision: 0.8733
Recall: 0.1139
F1-Score: 0.2015
ROC-AUC: 0.7696

              precision    recall  f1-score   support

           0       0.96      1.00      0.98    279823
           1       0.87      0.11      0.20     14768

    accuracy                           0.95    294591
   macro avg       0.91      0.56      0.59    294591
weighted avg       0.95      0.95      0.94    294591



In [10]:
# Random Forest
print("\n=== Training Random Forest ===")
rf_model = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1, max_depth=10)
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_rf):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\nTop 10 Important Features:\n{feature_importance.head(10)}")


=== Training Random Forest ===
Accuracy: 0.9554
Precision: 0.8502
Recall: 0.1337
F1-Score: 0.2311
ROC-AUC: 0.8164

Top 10 Important Features:
               feature  importance
2   Transaction Amount    0.483044
13    Account Age Days    0.352303
14    Transaction Hour    0.126219
3     Transaction Date    0.006484
1          Customer ID    0.004303
10          IP Address    0.004244
0       Transaction ID    0.004227
8    Customer Location    0.004131
12     Billing Address    0.004129
11    Shipping Address    0.004118


In [13]:
import time

# XGBoost
print("\n=== Training XGBoost ===")

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=7,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
)

# Mulai hitung waktu
start_time = time.perf_counter()

# Training model
xgb_model.fit(X_train, y_train)

# Selesai hitung waktu
xgb_time = time.perf_counter() - start_time

# Predict
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

# Evaluate
print(f"Training time: {xgb_time:.2f} seconds")
print(f"Training time: {xgb_time/60:.2f} minutes")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_xgb):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_xgb):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_xgb):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_xgb):.4f}")


=== Training XGBoost ===
Training time: 21.68 seconds
Training time: 0.36 minutes
Accuracy: 0.8092
Precision: 0.1657
Recall: 0.6954
F1-Score: 0.2677
ROC-AUC: 0.8211


In [7]:
# Compare models
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    'ROC-AUC': [
        roc_auc_score(y_test, y_pred_proba_lr),
        roc_auc_score(y_test, y_pred_proba_rf),
        roc_auc_score(y_test, y_pred_proba_xgb)
    ]
})

print("=== Model Comparison ===")
print(results.to_string(index=False))

# Best model
best_model_idx = results['F1-Score'].idxmax()
best_model_name = results.loc[best_model_idx, 'Model']
print(f"\n✓ Best Model: {best_model_name}")

=== Model Comparison ===
              Model  Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression  0.954751   0.873313 0.113895  0.201510 0.769597
      Random Forest  0.955392   0.850194 0.133735  0.231116 0.816394
            XGBoost  0.809227   0.165720 0.695423  0.267657 0.821108

✓ Best Model: XGBoost


In [8]:
# Save best model
if best_model_name == 'Logistic Regression':
    joblib.dump(lr_model, '../models/best_model.joblib')
    best_model = lr_model
elif best_model_name == 'Random Forest':
    joblib.dump(rf_model, '../models/best_model.joblib')
    best_model = rf_model
else:
    joblib.dump(xgb_model, '../models/best_model.joblib')
    best_model = xgb_model

# Save results
results.to_csv('../models/model_comparison.csv', index=False)

print("✓ Best model saved: models/best_model.joblib")
print("✓ Results saved: models/model_comparison.csv")

✓ Best model saved: models/best_model.joblib
✓ Results saved: models/model_comparison.csv
